In [ ]:
import pandas as pd

df = pd.read_csv('cinema.csv')
df.head(5)

In [ ]:
print('列ラベル：',df.columns)
print('行列サイズ',df.shape)

In [ ]:
df.plot(kind='scatter', x='SNS1', y='sales')
df.plot(kind='scatter', x='SNS2', y='sales')
df.plot(kind='scatter', x='actor', y='sales')
df.plot(kind='scatter', x='original', y='sales')

In [ ]:
#print('欠損値の確認')
#print(df.isnull())
print('\nanyメソッドで列単位で欠損値の有無を確認')
print(df.isnull().any(axis=0))
print('\n各列の欠損値の数を調べる')
print(df.isnull().sum())

In [ ]:
# 欠損値をその列の平均値で穴埋め
# df2 = df.fillna(df.mean())

# 欠損値をその列の中央値で穴埋め
df2 = df.fillna(df.median())
print('\n各列の欠損値の数を調べる')
print(df2.isnull().sum())

In [ ]:
for name in df2.columns:
    if name == 'cinema_id' or name =='sales':
        continue
    df2.plot(kind='scatter', x = name, y = 'sales')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
features = ['SNS1', 'SNS2', 'actor', 'sales']
# df_melt = df.melt(id_vars='target', value_vars=features, var_name='feature', value_name='value')
df_melt = df2.melt(value_vars=features, var_name='feature', value_name='value')

# 箱ひげ図を描画
plt.figure(figsize=(10,6))
# sns.boxplot(data=df_melt, x='feature', y='value', hue='target')
sns.boxplot(data=df_melt, x='feature', y='value')
plt.title('Dataset Boxplot')
plt.xlabel('Feature')
plt.ylabel('Value')
plt.legend(title='target')
plt.show()
plt.close

In [ ]:
# データフレームから特定の行の抜き出し方の確認
test = pd.DataFrame(
    {'Acolumn':[1,2,3],
     'Bcolumn':[4,5,6]}
)
print(test,'\n')
# print(test['Acolumn']<2,'\n')
print(test[(test['Acolumn']<2) | (test['Acolumn']>2)], '\n')
print(test.drop(0,axis=0), '\n')
print(test.drop('Bcolumn', axis=1), '\n')


In [ ]:
# 外れ値を削除する
no = df2[ ( df2['SNS2'] > 1000 ) & ( df2['sales'] < 8500 ) ].index
print('Index for Remove=',no)
print( df2[ ( df2['SNS2'] > 1000 ) & ( df2['sales'] < 8500 ) ])

df3 = df2.drop(no, axis=0)
df3.shape

In [ ]:
for name in df3.columns:
    if name == 'cinema_id' or name =='sales':
        continue
    df3.plot(kind='scatter', x = name, y = 'sales')

In [ ]:
# データフレームから特徴量と正解データを取り出す
col = ['SNS1', 'SNS2', 'actor', 'original']
x = df3[col]
t = df3['sales']
print('x=\n',x.head(3),'\nt=\n',t.head(3))

In [ ]:
# スライスを用いてデータフレームから特徴量と正解データを効率的に取り出す
x = df3.loc[: , 'SNS1':'original']
t = df3['sales']
print('x=\n',x.head(3),'\nt=\n',t.head(3))

In [ ]:
# ホールドアウト法により訓練データとテストデータに分割する
from sklearn.model_selection import train_test_split

# x_train, y_train：学習に利用する訓練データ
# x_test, y_test：検証に利用するテストデータ
# test_size　: 検証に利用する割合、注意：整数値を指定するとテストデータ件数とみなされる
x_train, x_test, y_train, y_test = train_test_split(x,t,test_size=0.2, random_state=0)
print("学習用訓練データの特徴量の行列サイズ：",x_train.shape)
print(x_train.head(3), "\n")
print("検証用テストデータの特徴量の行列サイズ：",x_test.shape)
print(x_test.head(3),"\n")
print("学習用訓練データの正解の行列サイズ：",y_train.shape)
print(y_train.head(3), "\n")
print("検証用テストデータの正解の行列サイズ：",y_test.shape)
print(y_test.head(3),"\n")

In [ ]:
# 重回帰モデルのLinearRegression関数をインポートする
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train,y_train)

In [ ]:
import numpy as np
print('平均値：\n',df3.mean().iloc[1:5])
print('中央値：\n',df3.median().iloc[1:5])
print('標準偏差：\n',df3.std().iloc[1:5])

print('Originalの平均を四捨五入：\n',np.round((df3.mean().iloc[4])))

In [ ]:
# 作成したモデルで興行収入の予測を行う
new = pd.DataFrame([[150,700,300,0]],columns=x_train.columns)
print(model.predict(new))


cinema_data = [[df3.mean().iloc[1],df3.mean().iloc[2],df3.mean().iloc[3],np.round(df3.mean().iloc[4])]]
print(cinema_data)
new = pd.DataFrame(cinema_data,columns=x_train.columns)
print(model.predict(new))

In [ ]:
# モデルのscoreを計算(決定係数R2を計算)
model.score(x_test,y_test)

In [ ]:
from sklearn.metrics import mean_absolute_error

pred = model.predict(x_test)

# 平均絶対誤差 MAE(Mean Absolute Error)を求める
mean_absolute_error(y_pred=pred,y_true=y_test)

In [ ]:
# モデルの保存
import pickle

with open('cinema.pkl','wb') as f:
    pickle.dump(model,f)

In [ ]:
# モデル式の係数と切片の確認
print(model.coef_)
print(model.intercept_)

tmp = pd.DataFrame(model.coef_)
tmp.index = x_train.columns
print(tmp)